In [ ]:
""" VERİ SETİNİ YÜKLEME """
import os
import polars as pl
import numpy as np

# Veriyi yükle
DATA_PATH = "/kaggle/input/datasets/denizbyat/metropt3-features/metropt3_features.parquet"

df = pl.read_parquet(DATA_PATH)

print(f"Satır sayısı : {df.shape[0]:,}")
print(f"Sütun sayısı : {df.shape[1]}")
print(f"RAM kullanımı: {df.estimated_size('mb'):.1f} MB")

Satır sayısı : 1,508,308 -- 
Sütun sayısı : 133 -- 
RAM kullanımı: 758.1 MB

In [ ]:
""" HAZIRLIK VE NORMALİZSAYON """
import numpy as np
from sklearn.preprocessing import StandardScaler

# timestamp ve is_suspect'i ayır
feature_cols = [col for col in df.columns 
                if col not in ['timestamp', 'is_suspect']]      # timestampi saten ayrı ayrı özellik olarak verdik, is_suspect ise hedef değişkenimiz

X = df.select(feature_cols).to_numpy()  # girdi (verdiğimiz özelliklerden çıkarım yapacak)
y = df['is_suspect'].to_numpy()     # cevap (doğru mu diye bakacak)

print(f"Feature matrix : {X.shape}")
print(f"Normal kayıt   : {(y == 0).sum():,} (%{(y == 0).mean()*100:.1f})")
print(f"Şüpheli kayıt  : {(y == 1).sum():,} (%{(y == 1).mean()*100:.1f})")

# Normalize ediyoruz çünkü veri setindeki özellikler farklı ölçeklerde olabilir ve bu, modelin öğrenmesini zorlaştırabilir.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nNormalizasyon tamamlandı")

Feature matrix : (1508308, 131) -- 
Normal kayıt   : 1,406,713 (%93.3) -- 
Şüpheli kayıt  : 101,595 (%6.7)

In [ ]:
""" ISOLATION FOREST MODELİ """
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=100,      # 100 karar ağacı
    contamination=0.067,   # Verimizdeki anomali oranı (%6.7) üstte belirttiğimiz gibi
    random_state=42,
    n_jobs=-1              # Tüm CPU çekirdeklerini kullan
)

print("Isolation Forest eğitiliyor")
iso_forest.fit(X_scaled)

# Tahmin — 1: normal, -1: anomali
iso_predictions = iso_forest.predict(X_scaled)
iso_predictions = (iso_predictions == -1).astype(int)  # -1 → 1 (anomali)

print("Eğitim tamamlandı ✓")
print(f"Anomali tespit edilen kayıt: {iso_predictions.sum():,} (%{iso_predictions.mean()*100:.1f})")

aşağıdaki Değerlendirme hücresine bakmamız gerekiyor.

isolation forest modelinin mantığı normal ve anomali durumlarının kümelenmesi mantığına dayanır. Mesela ormanda bulunan farklı toplumları düşünün. Bulunduğunuz topluluk ne kadar küçükse sizin adımlar atarak toplumunuzdan uzaklaşıp kaybolmanızda daha büyük bir toplumdan uzaklaşmanıza göre daha az sayıda adım gerektirip. Prensip temelde buna dayanır. Şüpheli durumlar küçük topluluk, Normal durumlar ise büyük toplumu ifade eder model de ona kontrol etmesi için verdiğimiz veriyi deneyerek ne kadar adımda kaybolduğuna bakarak bunu tespit eder.

In [ ]:
""" DEĞERLENDİRME """
from sklearn.metrics import classification_report, confusion_matrix

# precision, recall, f1-score, support metriklerini gösteren rapordur
print("=== ISOLATION FOREST SONUÇLARI ===\n")
print(classification_report(y, iso_predictions, 
                            target_names=['Normal', 'Şüpheli']))

# Gerçek ve tahmin edilen sınıfların karşılaştırıldığı tablodur
print("\nKarmaşıklık Matrisi:")
cm = confusion_matrix(y, iso_predictions)
print(f"                 Tahmin Normal  Tahmin Şüpheli")
print(f"Gerçek Normal  : {cm[0][0]:>12,}  {cm[0][1]:>14,}")
print(f"Gerçek Şüpheli : {cm[1][0]:>12,}  {cm[1][1]:>14,}")

Sonuçlar aşırı zayıf çıktı. örneğin f1: 0.08 değerinde. karmaşıklık matriside aynı şekilde iyi değil. şüpheli durumların %8 civarını yakalayabilmiş. Bunun sebebi muhtemelen veri setimizin gürültüsünün fazla olması. Sistem durmasından önce 2 saatlik pencereleri baz aldık ancak bunların çoğu bilerek bakım tarzında kapatmalar olabiliyor. Model doğru tahmin yapıyor ancak set fazla gürültülü